# MALTO Workshop 2026
## Workshop 1: Neural Networks

**Date:**

**Topic:** train a neural network for image classification on CIFAR-10, step by step.

This notebook follows the training loop of the lecture. Before the training, we choose four things:

| Choice | In the lecture | Section |
|---|---|---|
| **Data** | the training data, $n$ pairs $(x, y)$, and the validation data, which we do not use to update $\theta$ | 1 |
| **Model** | the architecture: the number of layers, their sizes, $\sigma$. It defines $g_\theta$ and the parameters $\theta$ | 2 |
| **Loss** | $\ell$, from the task: squared error for regression, cross-entropy for classification | 3 |
| **Optimizer** | SGD, Momentum or Adam, with its settings: $\eta$, $B$, the number of epochs $E$ | 4 |

Then we run the training loop (Section 5), we measure the final score on the test data (Section 6) and we look at the results (Section 7).

We train two models with the same data, loss and optimizer: an MLP (multilayer perceptron) and a CNN (convolutional neural network).

## 0. Setup

Use a GPU: **Runtime > Change runtime type > GPU**.

We use a fixed seed. Then the data split and the results are the same at each run.

In [ ]:
import copy
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

seed = 42
torch.manual_seed(seed)
rng = np.random.default_rng(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Data

> **Data:** the training data, $n$ pairs $(x, y)$, and the validation data, which we do not use to update $\theta$.

CIFAR-10 has 60,000 RGB images of 32x32 pixels in 10 classes.

- $x$ is one image: a tensor of shape $(3, 32, 32)$.
- $y$ is the class label: an integer from 0 to 9.

torchvision gives a ready `Dataset` class for CIFAR-10. A `Dataset` has two functions:

- `__len__` gives the number of pairs $n$.
- `__getitem__` gives the pair $(x, y)$ at index $i$.

Thus we do not write our own `Dataset` class.

### 1.1 Download

The first download has no transform, because we need the raw pixels to compute the statistics in Section 1.3. The download occurs one time for each Colab runtime.

In [ ]:
train_set_raw = datasets.CIFAR10(root="data", train=True, download=True)
test_set_raw = datasets.CIFAR10(root="data", train=False, download=True)
classes = train_set_raw.classes

print("Classes:", classes)
print("Official train images:", len(train_set_raw), "| Official test images:", len(test_set_raw))

### 1.2 Split: train, validation, test

| Set | Use |
|---|---|
| **Train** | We use it to update $\theta$, the model parameters. |
| **Validation** | We do **not** use it to update $\theta$. We use it to monitor the training and to select the hyperparameters. |
| **Test** | We use it one time only, at the end, for the final score. |

CIFAR-10 gives only a train set (50,000 images) and a test set (10,000 images). Thus we take 5,000 random images from the official train set for validation.

In [ ]:
n_val = 5000
perm = rng.permutation(len(train_set_raw))
val_idx = perm[:n_val].tolist()
train_idx = perm[n_val:].tolist()

print("Train:", len(train_idx), "| Validation:", len(val_idx), "| Test:", len(test_set_raw))

### 1.3 Transforms

**Normalization.** For each RGB channel, we subtract the mean and divide by the standard deviation. Then each channel has a mean of about 0 and a standard deviation of about 1. This makes the optimization with gradient descent more stable.

We compute the mean and the standard deviation on the **train split only**. If we use validation or test images here, information from them goes into the training (data leakage).

**Data augmentation (train only).** At each epoch, the model sees a slightly different version of each image:

- `RandomCrop(32, padding=4)` adds a border of 4 pixels, then cuts a random 32x32 area.
- `RandomHorizontalFlip()` mirrors the image left-right with probability 0.5.

This reduces overfitting: the model must learn the object, not the exact pixels.

Validation and test have **no augmentation**, because we want to measure the model on the real, unchanged images.

In [ ]:
train_pixels = train_set_raw.data[train_idx].astype(np.float32) / 255.0   # (45000, 32, 32, 3)
mean = tuple(float(m) for m in train_pixels.mean(axis=(0, 1, 2)))
std = tuple(float(s) for s in train_pixels.std(axis=(0, 1, 2)))
del train_pixels
print("Mean:", np.round(mean, 4), "| Std:", np.round(std, 4))

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),              # PIL image (H, W, C) in [0, 255] -> tensor (C, H, W) in [0, 1]
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])


def denormalize(img):
    # Undo the normalization to show an image: x * std + mean.
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (img.cpu() * s + m).clamp(0, 1).permute(1, 2, 0)   # (C, H, W) -> (H, W, C)

### 1.4 Datasets and DataLoaders

Train and validation come from the same official train set, but they need different transforms. Thus we load the train set two times (from the disk, no new download) and select the indices with `Subset`.

The `DataLoader` groups the pairs $(x, y)$ in mini-batches of $B$ images. At each step of gradient descent, we compute the loss and the gradient on one mini-batch, not on all $n$ images. For the train set, `shuffle=True` gives a new random order at each epoch.

In [ ]:
train_set_aug = datasets.CIFAR10(root="data", train=True, transform=train_transform)
train_set_eval = datasets.CIFAR10(root="data", train=True, transform=eval_transform)

train_set = Subset(train_set_aug, train_idx)
val_set = Subset(train_set_eval, val_idx)
test_set = datasets.CIFAR10(root="data", train=False, transform=eval_transform)

batch_size = 128   # B
pin_memory = device.type == "cuda"

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=pin_memory)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=pin_memory)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=pin_memory)

print("Batches per epoch -> train:", len(train_loader),
      "| val:", len(val_loader), "| test:", len(test_loader))

### 1.5 Look at one mini-batch

In [ ]:
images, labels = next(iter(train_loader))
print("x batch shape:", tuple(images.shape))   # (B, 3, 32, 32)
print("y batch shape:", tuple(labels.shape))   # (B,)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(denormalize(images[i]))
    ax.set_title(classes[labels[i]], fontsize=9)
    ax.axis("off")
plt.show()

## 2. Model

> **Model:** the architecture: the number of layers, their sizes, $\sigma$. It defines $g_\theta$ and the parameters $\theta$.

The model is a function $g_\theta$. It maps an image $x$ to one score for each class.

### 2.1 MLP

Three choices set the architecture:

1. **The number of layers:** 4 layers, that is 3 hidden layers and 1 output layer.
2. **The size of each layer:** 3072 → 512 → 256 → 128 → 10.
3. **The activation $\sigma$:** ReLU, $\sigma(z) = \max(0, z)$.

Each `nn.Linear(in, out)` holds the parameters of one layer: the matrix $W_l$ of shape (out, in) and the vector $b_l$ of shape (out).

The MLP takes a vector, not an image. Thus the forward pass first flattens each image of shape $(3, 32, 32)$ to a vector of 3072 numbers. Then each layer computes:

$$z_l = W_l\, h_{l-1} + b_l \qquad h_l = \sigma(z_l)$$

The output layer has no $\sigma$. Its output is the vector of **logits**: one score for each of the 10 classes. We do not apply softmax in the model, because the loss function applies it (Section 3).

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3 * 32 * 32, 512)   # layer 1 (hidden): W_1 (512 x 3072), b_1 (512)
        self.fc2 = nn.Linear(512, 256)           # layer 2 (hidden): W_2 (256 x 512),  b_2 (256)
        self.fc3 = nn.Linear(256, 128)           # layer 3 (hidden): W_3 (128 x 256),  b_3 (128)
        self.fc4 = nn.Linear(128, 10)            # layer 4 (output): W_4 (10 x 128),   b_4 (10)
        self.sigma = nn.ReLU()

    def forward(self, x):
        x = x.flatten(start_dim=1)    # (B, 3, 32, 32) -> (B, 3072)

        z1 = self.fc1(x)              # z_1 = W_1 x + b_1      (pre-activation)
        h1 = self.sigma(z1)           # h_1 = sigma(z_1)       (activation)   -> (B, 512)

        z2 = self.fc2(h1)             # z_2 = W_2 h_1 + b_2
        h2 = self.sigma(z2)           # h_2 = sigma(z_2)                      -> (B, 256)

        z3 = self.fc3(h2)             # z_3 = W_3 h_2 + b_3
        h3 = self.sigma(z3)           # h_3 = sigma(z_3)                      -> (B, 128)

        logits = self.fc4(h3)         # z_4 = W_4 h_3 + b_4, no sigma         -> (B, 10)
        return logits

### 2.2 CNN

Same task, same data, different architecture. The MLP flattens the image at the start, thus it does not use the spatial relations between near pixels. The CNN keeps the image structure $(3, 32, 32)$ and does not flatten it until the classifier.

The components:

- `Conv2d`: a convolution layer. A small matrix (kernel, or filter) moves over the image. At each position, it multiplies the pixel values by the kernel values and adds the results. Each filter gives one feature map.
  - `kernel_size=3`: the kernel is 3x3.
  - `padding=1`: one border of zeros around the input. With $P = (K - 1)/2$, the output has the same size as the input.
  - `stride=1`: the kernel moves 1 pixel at each step.
- `ReLU`: the activation $\sigma$ after the convolution, as in the MLP.
- `BatchNorm2d`: normalizes each channel over the mini-batch, then applies a learned scale $\gamma$ and shift $\beta$.
- `MaxPool2d(kernel_size=2)`: keeps the maximum value of each 2x2 area. It divides the height and the width by 2.
- `Flatten` and `Linear`: at the end, the classifier changes the feature maps into a vector and gives the 10 logits.

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2),       # (B, 32, 32, 32) -> (B, 32, 16, 16)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),       # -> (B, 64, 8, 8)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),       # -> (B, 128, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                      # -> (B, 2048)
            nn.Linear(128 * 4 * 4, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        logits = self.classifier(x)
        return logits

### 2.3 The parameters $\theta$

$\theta$ is the set of all the parameters of the model.

- **MLP:** all the $W_l$ and $b_l$. In PyTorch, `fc1.weight` is $W_1$ and `fc1.bias` is $b_1$, and so on to `fc4`.
- **CNN:** `Conv2d`, `BatchNorm2d` and `Linear` have parameters. `ReLU` and `MaxPool2d` have none. For `BatchNorm2d`, `weight` is $\gamma$ (scale) and `bias` is $\beta$ (shift).

When we create a model, PyTorch sets $\theta$ to small random values. The models do not know anything yet: the training (Section 5) changes $\theta$.

Both models take the same input and give the same output shape: $(B, 3, 32, 32) \to (B, 10)$.

In [ ]:
def show_parameters(model, model_name):
    print(f"===== {model_name} =====")
    print(model)
    print()
    n_params = 0
    for name, p in model.named_parameters():
        print(f"{name:24s} shape = {str(tuple(p.shape)):18s} count = {p.numel():,}")
        n_params += p.numel()
    print(f"Total number of parameters in theta: {n_params:,}\n")
    return n_params


mlp = MLP().to(device)
cnn = CNN().to(device)

n_params_mlp = show_parameters(mlp, "MLP")
n_params_cnn = show_parameters(cnn, "CNN")
print(f"The MLP has {n_params_mlp / n_params_cnn:.1f} times the parameters of the CNN.")

x_batch, y_batch = images.to(device), labels.to(device)
with torch.no_grad():
    print("MLP output shape:", tuple(mlp(x_batch).shape))
    print("CNN output shape:", tuple(cnn(x_batch).shape))

## 3. Loss

> **Loss:** $\ell$, from the task: squared error for regression, cross-entropy for classification.

Our task is classification with 10 classes. Thus we use the **cross-entropy** loss. (For regression, we use the squared error: `nn.MSELoss`.)

For one image, the model gives the logits $z$ (10 scores). The softmax changes the scores into 10 probabilities with sum 1. The loss is:

$$\ell = -\log\big(\mathrm{softmax}(z)_y\big)$$

$\ell$ is small when the model gives a high probability to the true class $y$.

`nn.CrossEntropyLoss` applies the softmax itself. Thus the models return logits, not probabilities. The result is the mean of $\ell$ over the $B$ images of the mini-batch: this is the loss $L$ on the batch.

**Sanity check.** Before the training, $\theta$ is random. Thus the model gives about the same probability to each class, $1/10$, and $\ell \approx -\log(1/10) = \log 10 \approx 2.30$.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

with torch.no_grad():
    print(f"Expected loss before training: {np.log(10):.3f}")
    print(f"MLP loss on one batch:         {loss_fn(mlp(x_batch), y_batch).item():.3f}")
    print(f"CNN loss on one batch:         {loss_fn(cnn(x_batch), y_batch).item():.3f}")

## 4. Optimizer

> **Optimizer:** SGD, Momentum or Adam, with its settings: $\eta$, $B$, the number of epochs $E$.

The optimizer is the algorithm that updates $\theta$ with the gradient $\nabla L$:

| Optimizer | Update |
|---|---|
| **SGD** | $\theta \leftarrow \theta - \eta\, \nabla L$ |
| **Momentum** | $v \leftarrow \mu\, v + \nabla L, \qquad \theta \leftarrow \theta - \eta\, v$ |
| **Adam** | keeps a moving average of the gradient and of the squared gradient, and uses them to give each parameter its own step size |

The settings are the same for all the models, thus the comparison between the models is fair:

- $\eta$ (learning rate): the size of each update step. Example start values: 0.001 for Adam, 0.01 to 0.1 for SGD and Momentum.
- $B$ (batch size): set in Section 1.4 (`batch_size`).
- $E$ (epochs): the number of full passes over the training data.

Each model gets its own optimizer, because an optimizer updates only the $\theta$ of one model. We create the optimizers in Section 5.1, together with the models.

In [ ]:
optimizer_name = "adam"   # "sgd", "momentum" or "adam"
learning_rate = 1e-3      # eta
momentum = 0.9            # mu, used only by "momentum"
epochs = 10               # E

print(f"Optimizer: {optimizer_name} | eta = {learning_rate} | B = {batch_size} | E = {epochs}")


def make_optimizer(model):
    params = model.parameters()   # theta of this model
    if optimizer_name == "sgd":
        return torch.optim.SGD(params, lr=learning_rate)
    if optimizer_name == "momentum":
        return torch.optim.SGD(params, lr=learning_rate, momentum=momentum)
    if optimizer_name == "adam":
        return torch.optim.Adam(params, lr=learning_rate)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")

## 5. Training loop

The code follows the steps of the lecture:

1. **Initialize** $\theta$ with small random values. *(Section 5.1: a new model.)*

**For each epoch, $E$ times** *(`fit`)*:

2. **Shuffle** the training data and divide them into batches of $B$ examples. *(`train_loader`)*

   **For each batch** *(`train_one_epoch`)*:

3. **Forward pass:** $\hat{y} = g_\theta(x)$ for the $B$ examples, and the loss $L$ on the batch.
4. **Backward pass:** backpropagation gives $\partial L / \partial w$ for every weight and bias.
5. **Update:** the optimizer changes $\theta$ with the gradient.

6. **Validate:** the loss on the validation data estimates the risk $R(\theta)$ on new data. *(`evaluate`)*

**Stop** after $E$ epochs, or when the validation loss stops decreasing (early stopping). At the end, we keep the $\theta$ with the lowest validation loss.

Two details of PyTorch:

- `model.train()` and `model.eval()` set the training mode and the evaluation mode. Some layers (`BatchNorm2d` in the CNN) work differently in the two modes.
- PyTorch adds the new gradients to the old ones. Thus `optimizer.zero_grad()` sets the gradients to 0 before each backward pass.

In [ ]:
patience = 3   # early stopping: stop after 3 epochs without a lower validation loss


def train_one_epoch(model, optimizer):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0

    for x, y in train_loader:                  # 2. shuffled batches of B examples
        x, y = x.to(device), y.to(device)

        y_hat = model(x)                       # 3. forward pass: logits, (B, 10)
        loss = loss_fn(y_hat, y)               #    loss L on the batch

        optimizer.zero_grad()                  # 4. backward pass
        loss.backward()                        #    dL/dw for each w in theta, kept in w.grad

        optimizer.step()                       # 5. update theta

        # Statistics. theta changes during the epoch, thus this is an average over the epoch.
        total_loss += loss.item() * x.size(0)
        total_correct += (y_hat.argmax(dim=1) == y).sum().item()
        total_n += x.size(0)

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()   # no backward pass here, thus no gradients
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        total_loss += loss_fn(y_hat, y).item() * x.size(0)
        total_correct += (y_hat.argmax(dim=1) == y).sum().item()
        total_n += x.size(0)
    return total_loss / total_n, total_correct / total_n


def fit(model, optimizer, model_name):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):         # for each epoch, E times
        start = time.time()
        train_loss, train_acc = train_one_epoch(model, optimizer)   # steps 2 to 5
        val_loss, val_acc = evaluate(model, val_loader)             # 6. validate

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"[{model_name}] epoch {epoch:2d}/{epochs} | "
              f"train loss {train_loss:.3f} acc {train_acc:.3f} | "
              f"val loss {val_loss:.3f} acc {val_acc:.3f} | {time.time() - start:.0f} s")

        # Keep a copy of the best theta (the lowest validation loss).
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        # Stop: early stopping.
        if epochs_without_improvement >= patience:
            print(f"[{model_name}] Early stopping: no lower validation loss for {patience} epochs.")
            break

    model.load_state_dict(best_state)          # use the best theta
    return history

### 5.1 Train the two models

**Step 1: initialize $\theta$.** A new model gets new random $\theta$: PyTorch takes each weight from a small interval that depends on the layer size. We create the models again here, thus each run of this cell starts the training from the beginning.

In [ ]:
torch.manual_seed(seed)
mlp = MLP().to(device)
mlp_optimizer = make_optimizer(mlp)
mlp_history = fit(mlp, mlp_optimizer, "MLP")

print()

torch.manual_seed(seed)
cnn = CNN().to(device)
cnn_optimizer = make_optimizer(cnn)
cnn_history = fit(cnn, cnn_optimizer, "CNN")

### 5.2 Learning curves

Solid line: training. Dashed line: validation.

When the training loss goes down and the validation loss goes up, the model starts to overfit. Early stopping looks for this point.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h, color in [("MLP", mlp_history, "C0"), ("CNN", cnn_history, "C1")]:
    x_epochs = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(x_epochs, h["train_loss"], color=color, label=f"{name} train")
    axes[0].plot(x_epochs, h["val_loss"], color=color, linestyle="--", label=f"{name} validation")
    axes[1].plot(x_epochs, h["train_acc"], color=color, label=f"{name} train")
    axes[1].plot(x_epochs, h["val_acc"], color=color, linestyle="--", label=f"{name} validation")

axes[0].set_title("Loss")
axes[1].set_title("Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

## 6. Test

We use the test data only now, after all the choices: architecture, optimizer, $\eta$, $E$, and the best $\theta$ from validation. No choice used the test data, thus the test score is a fair estimate of the performance on new data.

In [ ]:
for name, model in [("MLP", mlp), ("CNN", cnn)]:
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"{name}: test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

## 7. Analysis of the results

### 7.1 Predictions on test images

The same 16 test images for the two models (`test_loader` has `shuffle=False`). Green title: correct prediction. Red title: wrong prediction.

In [ ]:
x_show, y_show = next(iter(test_loader))
x_show, y_show = x_show[:16], y_show[:16]


@torch.no_grad()
def show_predictions(model, model_name):
    model.eval()
    pred = model(x_show.to(device)).argmax(dim=1).cpu()

    fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(denormalize(x_show[i]))
        correct = pred[i] == y_show[i]
        ax.set_title(f"pred: {classes[pred[i]]}\ntrue: {classes[y_show[i]]}",
                     fontsize=9, color="green" if correct else "red")
        ax.axis("off")
    fig.suptitle(f"{model_name}: {(pred == y_show).sum().item()}/16 correct")
    plt.show()


show_predictions(mlp, "MLP")
show_predictions(cnn, "CNN")

### 7.2 Confusion matrix

Row: true class. Column: predicted class.

The diagonal counts the correct predictions. A high value outside the diagonal shows two classes that the model mixes up. The test set has 1,000 images for each class, thus each row has a sum of 1,000.

The accuracy for each class is the number of correct predictions divided by the number of images of that class.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        all_pred.append(model(x.to(device)).argmax(dim=1).cpu())
        all_true.append(y)
    return torch.cat(all_pred).numpy(), torch.cat(all_true).numpy()


fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, model) in zip(axes, [("MLP", mlp), ("CNN", cnn)]):
    y_pred, y_true = predict(model, test_loader)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(name)

    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    print(f"{name} accuracy for each class:")
    for c, acc in zip(classes, per_class_acc):
        print(f"  {c:12s} {acc:.3f}")
    print()

plt.tight_layout()
plt.show()